# 07 · 自訂環境:把小遊戲包成 gymnasium

前六課都在用別人寫好的環境。這一課反過來:**自己刻一個**。只要實作 gymnasium 的標準介面,任何遊戲都能變成 RL 環境,接上 stable-baselines3 訓練。

我們做一個 **接水果(Catch)** 小遊戲:底部一塊木板,接住從上方掉下來的水果。麻雀雖小,五臟俱全——`observation_space`、`action_space`、`reset`、`step`、`render` 一個不少。

## 1. 安裝

In [ ]:
%pip install -q -U "gymnasium[classic-control]"

## 2. 一個 gymnasium 環境長什麼樣

繼承 `gym.Env`,定義五件事:
- `observation_space` / `action_space`:規格
- `reset()`:回到初始狀態,回 `(obs, info)`
- `step(action)`:走一步,回 `(obs, reward, terminated, truncated, info)`
- `render()`:畫出當前畫面

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class CatchEnv(gym.Env):
    """接水果：底部木板接住掉落的水果。這是我們親手刻的 gymnasium 環境。

    觀察 observation：[paddle_x, fruit_x, fruit_y]，三者都正規化到 0~1。
    動作 action：0=左移、1=不動、2=右移。
    獎勵 reward：接到 +1、漏接 -1，其餘步為 0。
    回合 episode：不自然結束，靠 TimeLimit 包裝器設定步數上限。
    """

    metadata = {"render_modes": ["ansi"]}

    def __init__(self, grid_size=5, render_mode=None):
        super().__init__()
        self.grid = grid_size
        self.render_mode = render_mode
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(3,), dtype=np.float32
        )

    def _obs(self):
        n = self.grid - 1
        return np.array(
            [self.paddle / n, self.fruit_x / n, self.fruit_y / n], dtype=np.float32
        )

    def _spawn_fruit(self):
        self.fruit_x = int(self.np_random.integers(0, self.grid))
        self.fruit_y = self.grid - 1

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.paddle = self.grid // 2
        self._spawn_fruit()
        return self._obs(), {}

    def step(self, action):
        # 木板依動作左右移（action-1 → -1/0/+1），夾在邊界內。
        self.paddle = int(np.clip(self.paddle + (action - 1), 0, self.grid - 1))
        self.fruit_y -= 1
        reward = 0.0
        if self.fruit_y <= 0:                       # 水果落到底了，結算這顆
            reward = 1.0 if self.paddle == self.fruit_x else -1.0
            self._spawn_fruit()                     # 立刻補一顆新的，遊戲繼續
        return self._obs(), reward, False, False, {}

    def render(self):
        rows = []
        for y in range(self.grid - 1, -1, -1):
            row = ["."] * self.grid
            if y == self.fruit_y:
                row[self.fruit_x] = "*"
            if y == 0:
                row[self.paddle] = "U"
            rows.append(" ".join(row))
        return "\n".join(rows)

## 3. 手動玩幾步,看 render

`U` 是木板,`*` 是水果。先用隨機動作跑,印出畫面。

In [ ]:
env = CatchEnv(grid_size=5)
obs, _ = env.reset(seed=0)
print("初始觀察 [paddle, fruit_x, fruit_y]:", obs)
print(env.render())
print("-" * 12)

for _ in range(4):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    print(env.render())
    print(f"reward={r}")
    print("-" * 12)

## 4. 用官方檢查器驗證介面

`check_env` 會幫你抓出 space、回傳值型別等常見錯誤——自訂環境必跑。

In [ ]:
from gymnasium.utils.env_checker import check_env

check_env(CatchEnv())      # 沒有報錯就代表介面合規
print("CatchEnv 通過 gymnasium 介面檢查 ✓")

## 小結

- 自訂環境 = 實作 `gym.Env` 的五件套:兩個 space + `reset` / `step` / `render`。
- 觀察設計要讓 agent「看得到做決定需要的資訊」(這裡:木板與水果的相對位置)。
- `check_env` 驗證介面合規,接下來才能交給任何演算法訓練。

下一課(壓軸):用 stable-baselines3 **訓練 agent 學會玩這個接水果遊戲**,並聊聊怎麼把它搬進瀏覽器。